# Covenant monitoring

**Maintenance** covenants are tested on a schedule (often quarterly); **incurrence** tests gate specific actions. **Springing** covenants activate when a revolver is drawn. Here we **project leverage** and compare it to a threshold under scenarios.

## Concept

We define **Net Debt / EBITDA** (sometimes defined differently in credit agreements). Forward quarters are evaluated; `evaluate_scenario_set` supplies **base vs downside** EBITDA paths. Compliance is a simple boolean flag in analysis code — production systems would map to **cure periods**, **baskets**, and **step-down** schedules.

In [ ]:
from finstack.statements import ModelBuilder, Evaluator, FinancialModelSpec
from finstack.statements_analytics import run_sensitivity, evaluate_scenario_set, goal_seek, trace_dependencies, explain_formula, run_variance, evaluate_dcf, run_corporate_analysis, pl_summary_report, credit_assessment_report
import json
print("Loaded finstack.statements + finstack.statements_analytics")

from finstack.statements_analytics import evaluate_scenario_set, trace_dependencies

PERIODS = ["2025Q1", "2025Q2", "2025Q3", "2025Q4", "2026Q1", "2026Q2"]


def z(v):
    return list(zip(PERIODS, v))


b = ModelBuilder("covenant-demo")
b.periods("2025Q1..2026Q2", "2025Q2")
b.value("revenue", z([100.0, 102.0, 105.0, 108.0, 110.0, 112.0]))
b.compute("ebitda", "revenue * 0.2")
b.value("total_debt", z([95.0, 93.0, 91.0, 89.0, 87.0, 85.0]))
b.value("cash", z([10.0, 10.5, 11.0, 11.5, 12.0, 12.5]))
b.compute("net_debt", "total_debt - cash")
b.compute("leverage", "net_debt / ebitda")

spec = b.build()
model_json = spec.to_json()
res = Evaluator().evaluate(spec)
print("Dependency tree for leverage:")
print(trace_dependencies(model_json, "leverage"))

covenant_limit = 3.75
print("Maintenance covenant: Net Debt / EBITDA <=", covenant_limit)
for p in PERIODS:
    lev = res.get("leverage", p)
    if lev is None:
        print(p, "leverage unavailable")
        continue
    ok = lev <= covenant_limit
    print(p, "leverage=", round(lev, 3), "compliant=" + str(ok))


In [ ]:
from finstack.statements import StatementResult

scenarios = json.dumps(
    {
        "scenarios": {
            "mgmt_case": {"overrides": {"revenue": 115.0}},
            "downside": {"overrides": {"revenue": 92.0}},
        }
    }
)
raw = evaluate_scenario_set(model_json, scenarios)
mp = json.loads(raw)
print("Scenario keys:", list(mp.keys()))
for name, rj in mp.items():
    r = StatementResult.from_json(json.dumps(rj))
    print("---", name, "---")
    for p in ("2026Q1", "2026Q2"):
        lev = r.get("leverage", p)
        if lev is None:
            print(p, "n/a")
            continue
        print(p, "leverage", round(lev, 3), "OK" if lev <= covenant_limit else "BREACH")
print("Springing example: if revolver drawn > 5, tighter 3.25x limit would apply — model as separate scenario.")


## Takeaways

- Encode **covenant definitions** explicitly (ratio nodes) so compliance is **repeatable** each quarter.
- **Scenarios** map cleanly to `evaluate_scenario_set` overrides on driver nodes like `revenue`.
- **Springing / incurrence** mechanics need richer event logic — use multiple models or boolean flags per policy text.